# S&P 500 Volatility Forecasting: GARCH Family vs. LSTM

**Authors:** Tanvika (Part 1: Data, GARCH, GUI) & Anvi (Part 2: LSTM, Evaluation, Notebook)  
**Dataset:** S&P 500 (`^GSPC`), daily OHLCV, 2000-01-01 to 2021-12-31  
**Target:** 5-Day Realized Volatility (`rv5`) — forward-looking, leakage-free  

This notebook is the complete, reproducible end-to-end pipeline.

---
**Runtime Flag:** `LOAD_SAVED = True` (default) skips re-training and loads saved `results/`. Set `False` to retrain from scratch (~1-2 minutes).

In [ ]:
# 0. Imports & Configuration
LOAD_SAVED = True  # Set False to retrain from scratch

import sys
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

warnings.filterwarnings('ignore')

ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import config
print('=== Project Configuration ===')
attrs = ['TICKER','START_DATE','END_DATE','TRAIN_END','VAL_END',
         'TEST_START','LOOKBACK','LSTM_UNITS','LSTM_DROPOUT','N_SEEDS',
         'GARCH_DIST','VOL_WINDOW']
for a in attrs:
    print(f'  {a:20s}: {getattr(config, a)}')

## Section 1: Data Pipeline & Exploratory Summary

`data.py` downloads S&P 500 OHLCV from Yahoo Finance (caches to `data/raw_gspc.csv`), computes features (`ret`, `hl_range`, `vol_chg`, `rv5`), assigns splits (`train` / `validation` / `test`), and saves `data/processed.csv`.

Key definitions:
- `ret_t = 100 x ln(close_t / close_{t-1})` — percent log-return
- `hl_range_t = 100 x ln(high_t / low_t)` — intraday range proxy
- `rv5_t = sqrt(mean(ret_{t-4..t}^2))` — 5-day realized volatility (**target**)

In [ ]:
# 1. Data Pipeline
from data import (download_data, prepare_data, verify_data,
                  calculate_statistics, create_figures)

processed_csv = config.DATA_DIR / 'processed.csv'

if not LOAD_SAVED or not processed_csv.exists():
    print('Running data.py pipeline...')
    raw = download_data()
    df_idx = prepare_data(raw)   # saves processed.csv; DatetimeIndex
    verify_data(df_idx)
    calculate_statistics(df_idx)
    create_figures(raw, df_idx)
    print('Done.')
else:
    print('Loading cached processed.csv ...')

# processed.csv stores date as the first column (index)
df = pd.read_csv(processed_csv, index_col=0, parse_dates=True)
df.index.name = 'date'

print(f'Shape: {df.shape}')
print('Samples per split:')
for split, grp in df.groupby('split'):
    start = grp.index.min().date()
    end   = grp.index.max().date()
    print(f'  {split:12s}: {len(grp):5,} rows  {start} to {end}')

assert df.isnull().sum().sum() == 0, 'NaNs in processed.csv!'
assert df.index.is_monotonic_increasing, 'Dates not monotonic!'
print('OK: no NaNs, dates monotonically increasing.')

In [ ]:
# 1b. Return Statistics
summary = pd.read_csv(config.RESULTS_DIR / 'data_summary.csv')
print('Return Statistics per Split:')
print(summary[['series','count','mean','std','skewness',
               'excess_kurtosis','arch_lm_pvalue']].to_string(index=False))

In [ ]:
# 1c. Data Figures (fig01-fig03)
for fname, title in [
    ('fig01_price.png',      'Fig 01: S&P 500 Close Price with Split Shading'),
    ('fig02_returns.png',    'Fig 02: Daily Log-Returns (Volatility Clustering)'),
    ('fig03_rv5_splits.png', 'Fig 03: 5-Day Realized Volatility by Split'),
]:
    p = config.FIG_DIR / fname
    if p.exists():
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.imshow(mpimg.imread(str(p)))
        ax.axis('off')
        ax.set_title(title, fontsize=11, fontweight='bold')
        plt.tight_layout(); plt.show()
    else:
        print(f'[INFO] {fname} not found.')

## Section 2: GARCH Family Models (GARCH, EGARCH, GJR-GARCH)

`garch_models.py` fits three GARCH variants on in-sample data (train + validation, up to 2019-12-31) with Student-t innovations.

**Rolling forecast scheme (strictly leakage-free):** For each block of `GARCH_REFIT_EVERY=20` test days, parameters are estimated on all prior returns (expanding window), then frozen. `sigma2_t` uses only returns through `t-1`.

**Prediction:** `pred_t = sqrt(mean(sigma2_{t-4..t}))` — directly comparable to `rv5`.

In [ ]:
# 2. GARCH Models
garch_ok = all(
    (config.RESULTS_DIR / f'preds_{m}.csv').exists()
    for m in ['garch', 'egarch', 'gjr']
)

if not LOAD_SAVED or not garch_ok:
    print('Running GARCH pipeline (~1 min)...')
    from garch_models import (fit_full_sample_params,
                               generate_garch_forecasts, next_day_forecast)
    fit_full_sample_params()
    for m in ['garch', 'egarch', 'gjr']:
        generate_garch_forecasts(m)
    next_day_forecast()
    print('Done.')
else:
    print('Loading saved GARCH results...')

params_df = pd.read_csv(config.RESULTS_DIR / 'garch_params.csv')
fit_stats  = pd.read_csv(config.RESULTS_DIR / 'garch_fit_stats.csv')
print('GARCH Fit Statistics (AIC/BIC):')
print(fit_stats.to_string(index=False))
print('\nAsymmetry Parameters (gamma):')
gamma_df = params_df[params_df['param'].str.contains('gamma', na=False)]
print(gamma_df.to_string(index=False))

In [ ]:
# 2b. GARCH Figures (fig04-fig05)
for fname, title in [
    ('fig04_news_impact.png',          'Fig 04: News Impact Curves'),
    ('fig05_garch_pred_vs_actual.png', 'Fig 05: GARCH Forecasts vs Actual RV5'),
]:
    p = config.FIG_DIR / fname
    if p.exists():
        fig, ax = plt.subplots(figsize=(13, 5))
        ax.imshow(mpimg.imread(str(p)))
        ax.axis('off')
        ax.set_title(title, fontsize=11, fontweight='bold')
        plt.tight_layout(); plt.show()
    else:
        print(f'[INFO] {fname} not found.')

## Section 3: LSTM Neural Network (PyTorch Ensemble)

**Architecture:** `LSTM(64) -> Dropout(0.2) -> Linear(1)` — PyTorch CPU.  
**Input:** 30 days of `[ret, hl_range, vol_chg]` from dates `t-30` to `t-1`. No feature from day `t` enters the window — strictly leakage-free.  
**Scaling:** `StandardScaler` (features) + `MinMaxScaler` (target) — both fit on **train split only**.  
**Ensemble:** Seeds 42, 43, 44 with `EarlyStopping(patience=10)`. Final prediction = average across seeds.

> **Note (docs/challenges_log.md):** TensorFlow failed on Windows Python 3.12 (hash mismatch on 350 MB wheel). PyTorch CPU used as approved fallback (Rule 8).

In [ ]:
# 3a. Build LSTM Sequences
from lstm_data import load_processed, make_sequences

proc_df   = load_processed()
data_dict = make_sequences(proc_df)

print('LSTM Dataset Splits:')
for split in ['train', 'val', 'test']:
    X = data_dict[f'X_{split}']
    y = data_dict[f'y_{split}']
    print(f'  {split:5s}: X.shape={X.shape}  y.shape={y.shape}')

assert data_dict['X_train'].shape[1:] == (config.LOOKBACK, 3), 'Wrong shape!'
print(f'OK: input shape (N, {config.LOOKBACK}, 3)')
print(f'Train scaler mean: {data_dict["scaler_x"].mean_.round(5)}')

In [ ]:
# 3b. Train or Load LSTM Ensemble
lstm_ok = (config.RESULTS_DIR / 'preds_lstm.csv').exists()

if not LOAD_SAVED or not lstm_ok:
    print('Training LSTM ensemble (seeds 42,43,44) ~45 s...')
    from lstm_model import train_lstm_ensemble, run_task_a3
    ensemble_models, scaler_x, scaler_y, dates_test = train_lstm_ensemble(data_dict)
    run_task_a3(ensemble_models, data_dict)
    print('Done.')
else:
    print('Loading saved LSTM results...')

history = pd.read_csv(config.RESULTS_DIR / 'lstm_history.csv')
print('Training History (best epoch per seed):')
best = history.loc[history.groupby('seed')['val_loss'].idxmin()]
print(best[['seed', 'epoch', 'train_loss', 'val_loss']].to_string(index=False))

p = config.FIG_DIR / 'fig06_lstm_loss.png'
if p.exists():
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.imshow(mpimg.imread(str(p)))
    ax.axis('off')
    ax.set_title('Fig 06: LSTM Loss Curves per Seed', fontsize=11, fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# 3c. LSTM Test Metrics
from metrics import rmse, mae, qlike

preds_lstm = pd.read_csv(config.RESULTS_DIR / 'preds_lstm.csv')
y_true = preds_lstm['y_true'].values
y_pred = preds_lstm['y_pred'].values

assert not np.any(np.isnan(y_pred)), 'NaNs in LSTM predictions!'
assert np.all(y_pred > 0), 'Non-positive LSTM predictions!'

print('LSTM Test Performance (2020-01-02 to 2021-12-31, 505 obs):')
print(f'  RMSE  : {rmse(y_true, y_pred):.6f}')
print(f'  MAE   : {mae(y_true, y_pred):.6f}')
print(f'  QLIKE : {qlike(y_true**2, y_pred**2, floor=config.EPS):.6f}')
print(f'  Range : {y_pred.min():.4f} to {y_pred.max():.4f} %')

p = config.FIG_DIR / 'fig07_lstm_pred_vs_actual.png'
if p.exists():
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.imshow(mpimg.imread(str(p)))
    ax.axis('off')
    ax.set_title('Fig 07: LSTM Ensemble vs Actual RV5', fontsize=11, fontweight='bold')
    plt.tight_layout(); plt.show()

## Section 4: Comparative Evaluation & Diebold-Mariano Test

`evaluate.py` computes:
1. **Naive benchmark:** `pred_t = rv5_{t-1}` (yesterday's rv5)
2. **Full test-set metrics:** RMSE, MAE, QLIKE for all 5 models
3. **Regime breakdown:** COVID stress (2020-02-15 to 2020-06-30) vs calm (2021)
4. **Diebold-Mariano test:** EGARCH vs LSTM on squared-error loss

QLIKE = `mean(y^2/p^2 - ln(y^2/p^2) - 1)` — penalizes under-prediction, critical in risk management.

In [ ]:
# 4. Evaluation
eval_ok = all(
    (config.RESULTS_DIR / f).exists()
    for f in ['metrics.csv','metrics_by_period.csv',
              'dm_test.csv','next_day_forecast.csv']
)

if not LOAD_SAVED or not eval_ok:
    print('Running evaluate.py...')
    from evaluate import main as run_evaluation
    run_evaluation()
    print('Done.')
else:
    print('Loading saved evaluation results...')

metrics_df = pd.read_csv(config.RESULTS_DIR / 'metrics.csv')
print('Full Test-Set Metrics (505 observations):')
print(metrics_df.to_string(index=False))
print()
for metric in ['rmse', 'mae', 'qlike']:
    if metric in metrics_df.columns:
        best = metrics_df.loc[metrics_df[metric].idxmin()]
        print(f'  Best {metric.upper():5s}: {best["model"].upper()} ({best[metric]:.6f})')

In [ ]:
# 4b. Regime Breakdown & Diebold-Mariano Test
period_df = pd.read_csv(config.RESULTS_DIR / 'metrics_by_period.csv')
print('Performance by Market Regime:')
print(period_df.to_string(index=False))

dm_df = pd.read_csv(config.RESULTS_DIR / 'dm_test.csv')
print('\nDiebold-Mariano Test (EGARCH vs LSTM):')
print(dm_df.to_string(index=False))
dm_p = float(dm_df['p_value'].iloc[0])
dm_d = float(dm_df['mean_loss_diff'].iloc[0])
sig  = 'YES (p<0.01)' if dm_p < 0.01 else ('YES (p<0.05)' if dm_p < 0.05 else 'NO')
print(f'\n  Significant: {sig}')
if dm_d > 0:
    print('  Positive loss differential: LSTM has lower MSE.')

In [ ]:
# 4c. Evaluation Figures (fig08-fig10)
for fname, title in [
    ('fig08_all_models_pred_vs_actual.png', 'Fig 08: All Models vs Actual RV5'),
    ('fig09_metric_bars.png',               'Fig 09: RMSE / MAE / QLIKE Bars'),
    ('fig10_covid_zoom.png',                'Fig 10: COVID Stress Period Zoom'),
]:
    p = config.FIG_DIR / fname
    if p.exists():
        fig, ax = plt.subplots(figsize=(13, 5))
        ax.imshow(mpimg.imread(str(p)))
        ax.axis('off')
        ax.set_title(title, fontsize=11, fontweight='bold')
        plt.tight_layout(); plt.show()
    else:
        print(f'[INFO] {fname} not found.')

## Section 5: Next-Day Forecast (Target: 2022-01-03)

All models generate a 5-day realized volatility forecast for the first trading day after dataset end.  
**Risk label** is derived from train-period rv5 percentiles.

In [ ]:
# 5. Next-Day Forecasts
nd_df = pd.read_csv(config.RESULTS_DIR / 'next_day_forecast.csv')
print('Next-Day Volatility Forecasts (2022-01-03):')
print(nd_df.sort_values('vol_forecast').to_string(index=False))

consensus = float(nd_df['vol_forecast'].mean())
train_rv5 = df.loc[df['split'] == 'train', 'rv5'].dropna()
p33 = float(np.percentile(train_rv5, 33.33))
p67 = float(np.percentile(train_rv5, 66.67))
risk = 'Low' if consensus < p33 else ('High' if consensus > p67 else 'Medium')
print(f'\n  Consensus: {consensus:.4f}%')
print(f'  Regime   : {risk}  (train p33={p33:.4f}%, p67={p67:.4f}%)')
print('  Forecasts for educational research only — not financial advice.')

## Section 6: Automated Test Suite

Running the full pytest suite:
- `test_garch_no_leakage.py` — GARCH invariance to future return replacement
- `test_lstm_windows.py` — correct sequences & train-only scaling
- `test_report_facts.py` — report numbers match results/*.csv exactly

In [ ]:
# 6. Automated Tests
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short'],
    capture_output=True, text=True, cwd=str(ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise AssertionError('One or more tests FAILED.')
print('All tests PASSED.')

## Section 7: Summary of Empirical Findings

### Full Test Set (505 days, 2020-2021)

| Model | RMSE | MAE | QLIKE | RMSE Rank | MAE Rank | QLIKE Rank |
|:------|:----:|:---:|:-----:|:---------:|:--------:|:----------:|
| **Naive** | **0.3146** | 0.1659 | 0.9029 | **1** | 2 | 2 |
| **LSTM** | 0.3415 | **0.1601** | **0.8413** | 2 | **1** | **1** |
| **EGARCH** | 0.5801 | 0.3425 | 1.0777 | 3 | 3 | 4 |
| **GJR-GARCH** | 0.5951 | 0.3711 | 1.0487 | 4 | 4 | 3 |
| **GARCH** | 0.6705 | 0.4234 | 1.0818 | 5 | 5 | 5 |

### Key Takeaways
1. **Regime Dependence:** LSTM dominated in calm 2021 (RMSE 0.1278 vs Naive 0.1679). Naive had lowest RMSE in the COVID shock (0.6272).
2. **Asymmetry Matters:** EGARCH and GJR both substantially outperformed symmetric GARCH(1,1).
3. **Statistical Superiority:** DM test: DM=5.41, p=6.26e-08 — LSTM significantly outperforms EGARCH on squared-error loss.

> No single model is always best. Report whatever the numbers show.